### Loading Test Data and Vocab

In [9]:
import torch
import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

from utils import load_config, load_vocab
from encoder import Encoder
from decoder import Decoder
from seq2Seq import Seq2Seq

In [3]:
config = load_config()
vocab = load_vocab()
vocab_reversed = load_vocab(rev=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [18]:
from utils import load_sets, get_dataloaders

train_set, val_set, test_set = load_sets()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set,
    val_set,
    test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

In [7]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_b = torch.load(
    "models/model_b/model_b_baseline.pt",
    map_location=device,
    weights_only=False
)

model_b.to(device)
model_b.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(13617, 256, padding_idx=0)
    (gru): GRU(256, 500, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(13617, 256, padding_idx=0)
    (gru): GRU(256, 500, num_layers=2, batch_first=True, dropout=0.4)
    (attn): Linear(in_features=500, out_features=500, bias=True)
    (context_linear): Linear(in_features=1000, out_features=500, bias=True)
    (linear): Linear(in_features=500, out_features=13617, bias=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
)

In [8]:
from inference import generate_response

In [11]:
prompts = [
    "how do i update ubuntu",
    "how do i remove a directory",
    "how do i extract a zip file",
    "how do i find my ip address",
    "how do i check ubuntu version",
    "what is apt",
    "what does sudo do",
    "how can i update my ubuntu system",
    "delete a folder in linux",
    "i am new to linux and want to update ubuntu safely"
]

qualitative_results = []

for p in prompts:
    text = p if p.startswith("<S0>") else f"<S0> {p}"
    
    out = generate_response(
        model=model_b,
        text=text,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )
    
    print("PROMPT:", p)
    print("OUTPUT:", out)
    print("-" * 60)

    qualitative_results.append({
        "prompt": p,
        "output": out
    })

PROMPT: how do i update ubuntu
OUTPUT: sudo apt-get update &&
------------------------------------------------------------
PROMPT: how do i remove a directory
OUTPUT: rm -rf -r
------------------------------------------------------------
PROMPT: how do i extract a zip file
OUTPUT: tar <UNK> <file>
------------------------------------------------------------
PROMPT: how do i find my ip address
OUTPUT: ifconfig your ifconfig
------------------------------------------------------------
PROMPT: how do i check ubuntu version
OUTPUT: dpkg -a <UNK>
------------------------------------------------------------
PROMPT: what is apt
OUTPUT: rips is <UNK>
------------------------------------------------------------
PROMPT: what does sudo do
OUTPUT: it <UNK> to
------------------------------------------------------------
PROMPT: how can i update my ubuntu system
OUTPUT: sudo apt-get update && sudo upgrade
------------------------------------------------------------
PROMPT: delete a folder in linux
O

In [12]:
import pandas as pd
qual_df = pd.DataFrame(qualitative_results)
qual_df

,prompt,output
0,how do i update ubuntu,sudo apt-get update &&
1,how do i remove a directory,rm -rf -r
2,how do i extract a zip file,tar <UNK> <file>
3,how do i find my ip address,ifconfig your ifconfig
4,how do i check ubuntu version,dpkg -a <UNK>
5,what is apt,rips is <UNK>
6,what does sudo do,it <UNK> to
7,how can i update my ubuntu system,sudo apt-get update && sudo upgrade
8,delete a folder in linux,i <UNK> folder
9,i am new to linux and want to update ubuntu sa...,i you have to a


### BLEU Evaluation

In [16]:
from bleu import compute_bleu

In [19]:
bleu_score = compute_bleu(
    model=model_b,
    loader=test_loader,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    n_batches=10
)

print("Model B BLEU:", bleu_score)

Computing BLEU for epoch...
  BLEU: batch 1/10...
  BLEU: batch 4/10...
  BLEU: batch 7/10...
  BLEU: batch 10/10...
Model B BLEU: 0.0025192541526038205


### Run Evaluation

### Comparison Table

In [21]:
summary_df = pd.DataFrame({
    "model": ["Model B"],
    "bleu": [bleu_score],
    "batches_evaluated": [10]
})

summary_df

,model,bleu,batches_evaluated
0,Model B,0.002519,10


### Showcasing Responses

In [22]:
prompts = [
    "how do i update ubuntu",
    "how do i remove a directory",
    "how do i extract a zip file",
    "how do i find my ip address",
    "how do i check ubuntu version",
    "what is apt",
    "what does sudo do",
    "how can i update my ubuntu system",
    "delete a folder in linux",
    "i am new to linux and want to update ubuntu safely"
]

qualitative_results = []

for p in prompts:
    text = p if p.startswith("<S0>") else f"<S0> {p}"
    
    out = generate_response(
        model=model_b,
        text=text,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )
    
    qualitative_results.append({
        "input": p,
        "model_b_prediction": out
    })

qual_df = pd.DataFrame(qualitative_results)
qual_df

,input,model_b_prediction
0,how do i update ubuntu,sudo apt-get update &&
1,how do i remove a directory,rm -rf -r
2,how do i extract a zip file,tar <UNK> <file>
3,how do i find my ip address,ifconfig your ifconfig
4,how do i check ubuntu version,dpkg -a <UNK>
5,what is apt,rips is <UNK>
6,what does sudo do,it <UNK> to
7,how can i update my ubuntu system,sudo apt-get update && sudo upgrade
8,delete a folder in linux,i <UNK> folder
9,i am new to linux and want to update ubuntu sa...,i you have to a


### Side by Side BLEU

In [28]:
def clean_text(text):
    return text.replace("<S0>", "").replace("<S1>", "").replace("<SEP>", "").strip()

In [32]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1

rows = []

for _, row in sample_test.iterrows():
    input_clean = clean_text(row["input"])
    ref_clean = clean_text(row["response"])

    text_for_model = row["input"] if row["input"].startswith("<S0>") else f"<S0> {input_clean}"

    pred = generate_response(
        model=model_b,
        text=text_for_model,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )

    pred_clean = clean_text(pred)

    sent_bleu = sentence_bleu(
        [ref_clean.split()],
        pred_clean.split(),
        smoothing_function=smooth
    )

    rows.append({
        "input": input_clean,
        "reference": ref_clean,
        "model_b_prediction": pred_clean,
        "model_b_sentence_bleu": sent_bleu
    })

comparison_df = pd.DataFrame(rows)
comparison_df.head(10)

,input,reference,model_b_prediction,model_b_sentence_bleu
0,question: is the new kde a lot better than gnome?,depends on your own feelings and needs,it's is <UNK>,0.000000
1,hello hello does anyone know how to change cu...,look for xrandr,sudo hostname <UNK>,0.000000
2,anyone here looking for easy money?,i usually work hard for mine.,i is you,0.041799
3,what is a good eviorment to learn python in ??,why not just the python interpreter?,<UNK> is honest,0.000000
4,anything better than crossover by code weavers...,good is great for questions like that,i <UNK> vlc,0.000000
5,hi there is wiki.ubuntu.com down?,"yes, seems to be",this is a,0.000000
6,"hi, is here any way to acellerate my flash pla...",you did mean hardware acceleration didnt you?,you adobe <UNK>,0.029950
7,can i put # in fstab? i just wanna uncomment a...,te # should be no problem,you it to,0.000000
8,"what is the best desktop for ubuntu: kde, gnom...",because you don't like gnome,opinions is <UNK>,0.000000
9,hello how can i install ubuntu and also keep ...,just install ubuntu after windows has been ins...,use need install,0.021460
